# Vinamilk 2011 scanned financial statements OCR diagnostic (v2)

Run one OCR engine at a time against the same three signed Vietnamese financial-statement pages. These pages come from one third-party preview packet and stay separate from the ten-region v7 benchmark. The labels include selected financial rows and visually checked balance-sheet value-cell locations; the source and review limitations are recorded in the dataset.

Attach the private Kaggle dataset `ltrngnhn/vinamilk-2011-vietnamese-scan-diagnostic-v2-assets`, enable Internet, and use a Kaggle GPU for EasyOCR/Surya. Keep `REQUIRE_GPU = True` for those GPU runs. Use the same render DPI and page labels for every engine.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

input_root = Path("/kaggle/input")
for bundle in [*input_root.glob("*/benchmark_project.zip"), *input_root.glob("datasets/*/*/benchmark_project.zip")]:
    extracted = Path("/kaggle/working/benchmark-assets/project")
    shutil.rmtree(extracted.parent, ignore_errors=True)
    shutil.unpack_archive(str(bundle), str(extracted.parent))
project_candidates = {
    *input_root.glob("*/project"),
    *input_root.glob("datasets/*/*/project"),
    *Path("/kaggle/working/benchmark-assets").glob("project"),
    *(entry.parent.parent for entry in input_root.rglob("scripts.processing.evaluate_ocr.py") if entry.parent.name == "scripts"),
}
assets_candidates = sorted(
    project for project in project_candidates
    if (project / "scripts" / "scripts.processing.evaluate_ocr.py").is_file()
)
if len(assets_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one attached OCR benchmark assets/project; "
        f"found: {assets_candidates}"
    )
assets = assets_candidates[0]
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
if repo.exists():
    shutil.rmtree(repo)
shutil.copytree(assets, repo)
%cd /kaggle/working/enterprise-knowledge-ai-agent


In [ ]:
ENGINE = "tesseract"  # Repeat with paddleocr, easyocr, or surya.
REQUIRE_GPU = False  # Set True for EasyOCR/Surya to require the Kaggle GPU.
DPI = 180
TESSERACT_PSM = 3  # Run a second Tesseract pass with PSM 11.
DATASET_PATH = "data/evaluation/vinamilk_ocr_ground_truth_2011_scan_diagnostic_v2.json"
DATASET_TAG = "2011-scan-diagnostic-v2"


In [ ]:
import subprocess, sys

# Install one engine only, avoiding incompatible OCR framework combinations.
subprocess.run([sys.executable, "-m", "pip", "install", "PyMuPDF==1.27.2.3", "Pillow==12.1.0"], check=True)
if ENGINE == "paddleocr":
    subprocess.run([sys.executable, "-m", "pip", "--default-timeout=300", "--retries=5", "install", "paddlepaddle==3.3.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cpu/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "paddleocr==3.7.0"], check=True)
elif ENGINE == "easyocr":
    subprocess.run([sys.executable, "-m", "pip", "install", "easyocr==1.7.2"], check=True)
elif ENGINE == "surya":
    subprocess.run([sys.executable, "-m", "pip", "install", "surya-ocr==0.17.1", "transformers==4.57.3"], check=True)
elif ENGINE == "tesseract":
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie"], check=True)
else:
    raise ValueError(f"Unknown ENGINE: {ENGINE}")


In [ ]:
import os

os.environ["HF_HOME"] = "/kaggle/working/model-cache/huggingface"
os.environ["TORCH_HOME"] = "/kaggle/working/model-cache/torch"
if ENGINE == "surya":
    # Fit Surya's detection/recognition batches within Kaggle's 15 GB Tesla T4.
    os.environ["DETECTOR_BATCH_SIZE"] = "16"
    os.environ["RECOGNITION_BATCH_SIZE"] = "128"
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
if ENGINE in {"easyocr", "surya"}:
    import torch
    cuda_available = torch.cuda.is_available()
    print("torch", torch.__version__, "CUDA", cuda_available)
    if REQUIRE_GPU and not cuda_available:
        raise RuntimeError("GPU requested for this OCR engine but CUDA is unavailable; restart with a Kaggle GPU accelerator.")
elif ENGINE == "paddleocr":
    import paddle
    print("paddle", paddle.__version__, "CUDA build", paddle.is_compiled_with_cuda(), "device", paddle.device.get_device())


In [ ]:
import json
dataset_path = Path(DATASET_PATH)
if not dataset_path.is_file():
    raise FileNotFoundError(f"Missing OCR labels: {dataset_path}")
dataset = json.loads(dataset_path.read_text(encoding="utf-8"))
for source in sorted({page["file"] for page in dataset["pages"]}):
    if not Path(source).is_file():
        raise FileNotFoundError(f"Missing benchmark source PDF: {source}")
output = Path("/kaggle/working") / f"vinamilk-ocr-{DATASET_TAG}-{ENGINE}.json"
command = [sys.executable, "scripts/processing/evaluate_ocr.py", "--dataset", DATASET_PATH, "--engines", ENGINE, "--dpi", str(DPI), "--output", str(output)]
if ENGINE == "tesseract":
    command.extend(["--tesseract-psm", str(TESSERACT_PSM)])
subprocess.run(command, check=True)
print(f"Download the result from Kaggle Outputs: {output}")


## Diagnostic limitations

These three pages are from one third-party preview packet, not issuer-hosted scans, and do not represent the broader Vietnamese OCR population. The production-parser evaluation separately covers 17 selected rows, 34 numeric cells, 6 period headers, and 20 visually checked value-cell boxes; those boxes came from the PDF text layer and were visually checked, not independently click-annotated. This engine benchmark reports rendered-text CER/WER and selected row/cell-text recall; it does not compare engine-generated cell coordinates. Do not pool these results into v7 or use them alone to select a production OCR engine. Record the engine/model version, runtime device, source PDF hash, and rendered image hashes. After downloading one report per engine, compare runs with `scripts/processing/compare_ocr_reports.py` and verify all rendered-image hashes match before comparing scores.
